In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/life_satisfaction_canadian_survey_ml.csv")

df_ml.head()

,Total_active_time,Total_physical_act_time,Other_physical_act_time,Physical_vigorous_act_time,Marital_status__missing,Household__missing,Worked_job_business__missing,Edu_level__missing,Gen_health_state__missing,Life_satisfaction__missing,Mental_health_state__missing,Stress_level__missing,Work_stress__missing,Sense_belonging__missing,Weight_state__missing,BMI_12_17__missing,BMI_18_above__missing,Sleep_apnea__missing,High_BP__missing,Diabetic__missing,Fatigue_syndrome__missing,Mood_disorder__missing,Anxiety_disorder__missing,Respiratory_chronic_con__missing,Musculoskeletal_con__missing,Cardiovascular_con__missing,Health_utility_indx__missing,Pain_status__missing,Act_improve_health__missing,Fruit_veg_con__missing,Smoked__missing,Tobaco_use__missing,weekly_alcohol__missing,Cannabies_use__missing,Drug_use__missing,Total_active_time__missing,Total_physical_act_time__missing,Other_physical_act_time__missing,Physical_vigorous_act_time__missing,Work_hours__missing,working_status__missing,Aboriginal_identity__missing,Birth_country__missing,Immigrant__missing,Insurance_cover__missing,Food_security__missing,Income_source__missing,Total_income__missing,Health_region_ grouped_10912,Health_region_ grouped_10913,Health_region_ grouped_11900,Health_region_ grouped_12901,Health_region_ grouped_12902,Health_region_ grouped_12903,Health_region_ grouped_12904,Health_region_ grouped_13901,Health_region_ grouped_13902,Health_region_ grouped_13903,Health_region_ grouped_13904,Health_region_ grouped_13906,Health_region_ grouped_24901,Health_region_ grouped_24902,Health_region_ grouped_24903,Health_region_ grouped_24904,Health_region_ grouped_24905,Health_region_ grouped_24906,Health_region_ grouped_24907,Health_region_ grouped_24908,Health_region_ grouped_24909,Health_region_ grouped_24911,Health_region_ grouped_24912,Health_region_ grouped_24913,Health_region_ grouped_24914,Health_region_ grouped_24915,Health_region_ grouped_24916,Health_region_ grouped_35926,Health_region_ grouped_35927,Health_region_ grouped_35930,Health_region_ grouped_35933,Health_region_ grouped_35934,Health_region_ grouped_35935,Health_region_ grouped_35936,Health_region_ grouped_35937,Health_region_ grouped_35938,Health_region_ grouped_35939,Health_region_ grouped_35940,Health_region_ grouped_35941,Health_region_ grouped_35942,Health_region_ grouped_35943,Health_region_ grouped_35944,Health_region_ grouped_35946,Health_region_ grouped_35947,Health_region_ grouped_35949,Health_region_ grouped_35951,Health_region_ grouped_35953,Health_region_ grouped_35955,Health_region_ grouped_35957,Health_region_ grouped_35958,Health_region_ grouped_35960,Health_region_ grouped_35961,Health_region_ grouped_35962,Health_region_ grouped_35965,Health_region_ grouped_35966,Health_region_ grouped_35968,Health_region_ grouped_35970,Health_region_ grouped_35975,Health_region_ grouped_35995,Health_region_ grouped_46901,Health_region_ grouped_46902,Health_region_ grouped_46903,Health_region_ grouped_46905,Health_region_ grouped_47901,Health_region_ grouped_47904,Health_region_ grouped_47905,Health_region_ grouped_47906,Health_region_ grouped_47907,Health_region_ grouped_47909,Health_region_ grouped_48931,Health_region_ grouped_48932,Health_region_ grouped_48933,Health_region_ grouped_48934,Health_region_ grouped_48935,Health_region_ grouped_59911,Health_region_ grouped_59912,Health_region_ grouped_59913,Health_region_ grouped_59914,Health_region_ grouped_59921,Health_region_ grouped_59922,Health_region_ grouped_59923,Health_region_ grouped_59931,Health_region_ grouped_59932,Health_region_ grouped_59933,Health_region_ grouped_59941,Health_region_ grouped_59942,Health_region_ grouped_59943,Health_region_ grouped_59951,Health_region_ grouped_59952,Health_region_ grouped_60901,Gender_2,Marital_status_2,Marital_status_6,Household_2,Age_2,Age_3,Age_4,Age_5,Worked_job_business_2,Edu_level_2,Edu_level_3,Gen_health_state_2,Gen_health_state_3,Gen_health_state_4,Gen_health_state_5,Life_satisfaction_1,Life_satisf

In [7]:
# Reconstruct Life Satisfaction from one-hot encoded columns
life_cols = [f"Life_satisfaction_{i}" for i in range(1, 11)]

df = df_ml.copy()

# Step 1: reconstruct categories 1..10 from dummy columns
df["Life_satisfaction"] = np.select(
    [df[col] == 1 for col in life_cols],
    list(range(1, 11)),
    default=0   # dropped reference category from drop_first=True
)

# Step 2: restore true missing targets
df.loc[df["Life_satisfaction__missing"] == 1, "Life_satisfaction"] = np.nan

# Check result
print(df["Life_satisfaction"].value_counts(dropna=False).sort_index())
print("Missing values in target:", df["Life_satisfaction"].isna().sum())

Life_satisfaction
0.0       471
1.0       226
2.0       455
3.0       785
4.0      1297
5.0      5566
6.0      5426
7.0     15019
8.0     32001
9.0     19835
10.0    22971
NaN      4200
Name: count, dtype: int64
Missing values in target: 4200


In [8]:
# Drop rows with missing target
df = df.dropna(subset=["Life_satisfaction"]).copy()

# Drop original target dummy columns and missing indicator
df = df.drop(columns=life_cols + ["Life_satisfaction__missing"])

# Define predictors and target
X = df.drop(columns=["Life_satisfaction"])
y = df["Life_satisfaction"]

print(X.shape, y.shape)
print("Missing values in target:", y.isna().sum())

(104052, 387) (104052,)
Missing values in target: 0


In [9]:
# Split data (regression - no stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (83241, 387)
Test shape: (20811, 387)


In [10]:
# Random Forest for feature selection
rf_selector = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Number of selected features:", len(selected_features))

Number of selected features: 194


In [11]:
# Linear Regression on selected features
linreg_hybrid = LinearRegression()

linreg_hybrid.fit(X_train_sel, y_train)
y_pred_hybrid1 = linreg_hybrid.predict(X_test_sel)

print("Hybrid 1: RF -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)))
print("R2:", r2_score(y_test, y_pred_hybrid1))

Hybrid 1: RF -> Linear Regression
MAE: 0.9491934964881887
RMSE: 1.2829405891408434
R2: 0.4091217995611235


In [12]:
# Interpret coefficients
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": linreg_hybrid.coef_
})

coef_df["Abs"] = coef_df["Coefficient"].abs()
coef_df.sort_values("Abs", ascending=False).head(15)

,Feature,Coefficient,Abs
119,Mental_health_state_5,-2.414443,2.414443
115,Gen_health_state_5,-1.903794,1.903794
118,Mental_health_state_4,-1.428427,1.428427
123,Stress_level_5,-1.319281,1.319281
114,Gen_health_state_4,-0.991659,0.991659
122,Stress_level_4,-0.832507,0.832507
8,Gen_health_state__missing,-0.795782,0.795782
117,Mental_health_state_3,-0.783778,0.783778
9,Mental_health_state__missing,-0.767824,0.767824
130,Sense_belonging_4,-0.681395,0.681395


In [13]:
xgb_selector = XGBRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector.fit(X_train, y_train)

selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

selector_xgb.fit(X_train, y_train)

X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

selected_features_xgb = X_train.columns[selector_xgb.get_support()]

In [14]:
linreg_hybrid_xgb = LinearRegression()

linreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)
y_pred_hybrid2 = linreg_hybrid_xgb.predict(X_test_sel_xgb)

print("Hybrid 2: XGB -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)))
print("R2:", r2_score(y_test, y_pred_hybrid2))

Hybrid 2: XGB -> Linear Regression
MAE: 0.9491147174373631
RMSE: 1.283244155612506
R2: 0.4088421419789039


In [15]:
base_estimators = [
    ("lr", LinearRegression()),
    ("rf", RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Hybrid 3: Stacking")
print("MAE:", mean_absolute_error(y_test, y_pred_stack))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_stack)))
print("R2:", r2_score(y_test, y_pred_stack))

Hybrid 3: Stacking
MAE: 0.9463929933459769
RMSE: 1.279663884845614
R2: 0.41213621917778387


In [16]:
results = pd.DataFrame([
    {
        "Model": "RF -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid1),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)),
        "R2": r2_score(y_test, y_pred_hybrid1),
    },
    {
        "Model": "XGB -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid2),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)),
        "R2": r2_score(y_test, y_pred_hybrid2),
    },
    {
        "Model": "Stacking",
        "MAE": mean_absolute_error(y_test, y_pred_stack),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_stack)),
        "R2": r2_score(y_test, y_pred_stack),
    }
])

results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
2,Stacking,0.946393,1.279664,0.412136
0,RF -> Linear Regression,0.949193,1.282941,0.409122
1,XGB -> Linear Regression,0.949115,1.283244,0.408842
